# Testing Scraper on GDELT Data

### Imports and Notebook Settings


In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
import os 
import re
from pathlib import Path

In [9]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
# import seaborn as sns 
import plotly.express as px
import pprint 

In [10]:
import requests
import trafilatura
from bs4 import BeautifulSoup
from datetime import datetime, timezone

In [11]:
ROOT = Path.cwd().parent

In [6]:
style_path = ( ROOT 
              / 'notebooks' 
              / 'styler.mplstyle'
              )
plt.style.use(style_path)

### Test scaping Methods

In [7]:
data_path = (ROOT 
             / 'data'
             / 'silver'
             / 'source=gdelt'
             /  'records.parquet'
             )

In [8]:
df = pd.read_parquet(data_path)

In [9]:
df = df.drop(columns=['raw'])

In [10]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata'],
      dtype='object')

In [11]:
for i in df['title'].head(30):
    print(i)

Ranchers discuss lasting impacts from wildfires, status of Xcel claims
Gas leak closes part of Harrison Drive in Clinton
Xcel Energy offers bill assistance as Minnesota Cold Weather Rule ends
WisBusiness: the Show with Brent Ridge, Dairyland Power Cooperative
Perkins Road Overpass area businesses contracting towing companies due to parking congestion
Squeeze the Day: Amarillo, Texas Hosts Lemonade Day Festivities
Southern Company reports first-quarter 2025 earnings
Entergy Corporation: Energy for a Better Future: Entergy's Growth and Resiliency Strategy
Southern : 1Q25 Earnings Package | MarketScreener
Alliant Energy Offers Dividend Growth Amid Market Uncertainty | Investor's Business Daily
Weekend events in the 53581 and beyond
'Fox in charge of the hen house': Judge with controversial ties to Duke nominated to Utilities Commission
Are Utilities Prepped To Deal With A Slew Of Hurricanes And Wildfires?
Squeeze the Day: Amarillo, Texas Hosts Lemonade Day Festivities
The Southern Company

In [12]:
pprint.pprint(df.iloc[3])

record_id                                      20250501050000-193
source                                                      gdelt
source_type                                                   api
title           WisBusiness: the Show with Brent Ridge, Dairyl...
text                                                             
published_at                            2025-05-01T05:00:00+00:00
retrieved_at                     2026-06-19T23:25:44.981825+00:00
url             https://www.wisbusiness.com/2025/wisbusiness-t...
region                                                         US
categories      [TAX_FNCACT, TAX_FNCACT_CHIEF, TAX_FNCACT_EXEC...
metadata        {'source_common_name': 'wisbusiness.com', 'gde...
Name: 3, dtype: object


In [13]:
import nest_asyncio
nest_asyncio.apply()


In [14]:
from datetime import datetime, timezone
import json

import aiohttp
import trafilatura
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


def categorize_status(status: int | None) -> str:
    if status == 403:
        return "forbidden"
    if status == 404:
        return "not_found"
    if status == 429:
        return "rate_limited"
    if status is not None and 500 <= status < 600:
        return "server_error"
    if status is not None:
        return "http_error"
    return "unknown"


async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    timeout: int = 15,
) -> dict:
    try:
        async with session.get(url, timeout=timeout) as response:
            text = await response.text(errors="ignore")

            if response.status >= 400:
                return {
                    "success": False,
                    "html": None,
                    "status_code": response.status,
                    "error_type": categorize_status(response.status),
                    "error_message": f"HTTP {response.status}",
                    "method": "aiohttp",
                }

            return {
                "success": True,
                "html": text,
                "status_code": response.status,
                "error_type": None,
                "error_message": None,
                "method": "aiohttp",
            }

    except TimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "timeout",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except aiohttp.ClientConnectionError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "connection_error",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "unknown",
            "error_message": str(exc),
            "method": "aiohttp",
        }


async def fetch_html_with_playwright(
    url: str,
    timeout: int = 30000,
) -> dict:
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)

            page = await browser.new_page(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/137.0.0.0 Safari/537.36"
                )
            )

            response = await page.goto(
                url,
                wait_until="networkidle",
                timeout=timeout,
            )

            html = await page.content()
            status_code = response.status if response else None

            await browser.close()

        return {
            "success": True,
            "html": html,
            "status_code": status_code,
            "error_type": None,
            "error_message": None,
            "method": "playwright",
        }

    except PlaywrightTimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_timeout",
            "error_message": str(exc),
            "method": "playwright",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_error",
            "error_message": str(exc),
            "method": "playwright",
        }


def extract_with_trafilatura(
    html: str,
    url: str,
    extractor: str,
) -> dict:
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=False,
    )

    if result is None:
        return {
            "success": False,
            "extractor": extractor,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "error_type": "parse_failed",
            "error_message": "trafilatura returned None",
        }

    parsed = json.loads(result)
    text = parsed.get("text", "") or ""

    return {
        "success": True,
        "extractor": extractor,
        "title": parsed.get("title"),
        "author": parsed.get("author"),
        "date": parsed.get("date"),
        "text": text,
        "error_type": None,
        "error_message": None,
    }


async def scrape_article(
    session: aiohttp.ClientSession,
    url: str,
) -> dict:
    retrieved_at = datetime.now(timezone.utc).isoformat()

    fetch_result = await fetch_html(session, url)

    if (
        not fetch_result["success"]
        and fetch_result["error_type"] == "forbidden"
    ):
        fetch_result = await fetch_html_with_playwright(url)

    if not fetch_result["success"]:
        return {
            "success": False,
            "url": url,
            "status_code": fetch_result["status_code"],
            "error_type": fetch_result["error_type"],
            "error_message": fetch_result["error_message"],
            "fetch_method": fetch_result["method"],
            "extractor": None,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "text_length": 0,
            "retrieved_at": retrieved_at,
        }

    extracted = extract_with_trafilatura(
        html=fetch_result["html"],
        url=url,
        extractor=f"trafilatura_after_{fetch_result['method']}",
    )

    return {
        "success": extracted["success"],
        "url": url,
        "status_code": fetch_result["status_code"],
        "error_type": extracted["error_type"],
        "error_message": extracted["error_message"],
        "fetch_method": fetch_result["method"],
        "extractor": extracted["extractor"],
        "title": extracted["title"],
        "author": extracted["author"],
        "date": extracted["date"],
        "text": extracted["text"],
        "text_length": len(extracted["text"]),
        "retrieved_at": retrieved_at,
    }

In [15]:
import pandas as pd
import aiohttp
import asyncio


async def scrape_urls(urls: list[str]) -> pd.DataFrame:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }

    async with aiohttp.ClientSession(headers=headers) as session:
        results = [
            await scrape_article(session, url)
            for url in urls
        ]

    return pd.DataFrame(results)



In [16]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata'],
      dtype='object')

In [17]:
urls = df["url"].dropna().head(100).tolist()

results_df = await scrape_urls(urls)

CancelledError: 

In [ ]:
results_df[
    [
        "success",
        "url",
        "title",
        "date",
        "text_length",
        "extractor",
    ]
]

,success,url,title,date,text_length,extractor
0,True,https://www.eenews.net/articles/the-absolute-e...,‘The absolute edge of precedent’: FERC prepare...,2026-04-20,13020,trafilatura_after_aiohttp
1,True,https://www.themarketsdaily.com/2026/04/20/ame...,American Electric Power (NASDAQ:AEP) Price Tar...,2026-04-20,5933,trafilatura_after_aiohttp
2,True,https://www.12newsnow.com/article/traffic/wrec...,Access Denied,None,302,trafilatura_after_playwright
3,True,https://www.insurancejournal.com/news/midwest/...,Michigan Feared Cheboygan Dam Danger Before Ra...,2026-04-20,9263,trafilatura_after_aiohttp
4,True,https://www.benzinga.com/Opinion/26/04/5192303...,Duke Energy: 100-Year Dividend Track Record — ...,2026-04-20,4349,trafilatura_after_aiohttp
...,...,...,...,...,...,...
95,True,https://wbt.com/1644984/appeals-court-wrestles...,Appeals Court wrestles with NC permit for MVP ...,2026-04-28,10223,trafilatura_after_aiohttp
96,False,https://wwmt.com/news/local/consumers-energy-d...,None,None,0,None
97,True,https://wwmt.com/news/local/consumers-energy-d...,Consumers Energy deploys 500 crews as outages ...,2026-04-28,4071,trafilatura_after_aiohttp
98,True,https://www.fool.com/investing/2026/04/29/ener...,Energy Stock Showdown: NextEra or Duke Energy ...,2026-04-29,3437,trafilatura_after_aiohttp


In [ ]:
success_rate = results_df['success'].sum() / len(results_df)

print(f"{success_rate * 100}% of articles could be scraped")

90.0% of articles could be scraped


In [ ]:
results_df["error_type"].value_counts(dropna=False)

error_type
None                  90
not_found              5
unknown                2
rate_limited           2
playwright_timeout     1
Name: count, dtype: int64

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(
        results_df.loc[
            ~results_df["success"],
            [
                "url",
                "status_code",
                "error_type",
                "error_message",
            ],
        ]
    )

,url,status_code,error_type,error_message
7,https://www.dailypolitical.com/2026/04/21/analysts-recent-ratings-updates-for-american-electric-power-aep.html,404.0,not_found,HTTP 404
8,https://www.dailypolitical.com/2026/04/21/truist-financial-begins-coverage-on-american-electric-power-nasdaqaep.html,404.0,not_found,HTTP 404
30,https://www.erienewsnow.com/news/state/ohio-supreme-court-guts-submetering-business-said-to-drive-up-renters-electric-bills/article_7e99e81f-249c-5e71-80e5-5ba1c2aeb0b0.html,404.0,not_found,HTTP 404
44,https://www.powermag.com/duke-energys-robinson-nuclear-plant-gets-nrc-approval-to-operate-until-2050/,NaN,playwright_timeout,"Page.goto: Timeout 30000ms exceeded.\nCall log:\n - navigating to ""https://www.powermag.com/duke-energys-robinson-nuclear-plant-gets-nrc-approval-to-operate-until-2050/"", waiting until ""networkidle""\n"
59,https://finance.yahoo.com/markets/stocks/articles/southern-company-raises-dividend-2-141743349.html,NaN,unknown,"400, message='Got more than 8190 bytes when reading: b""connect-src \'self\' wss://*.finance.yahoo.com/ https://*.cdn.yimg.com https://*.oath.com https://*.ya..."".', url='https://finance.yahoo.com/markets/stocks/articles/southern-company-raises-dividend-2-141743349.html'"
60,https://finance.yahoo.com/sectors/energy/articles/nrc-renews-license-duke-energy-141832333.html,NaN,unknown,"400, message='Got more than 8190 bytes when reading: b""connect-src \'self\' wss://*.finance.yahoo.com/ https://*.cdn.yimg.com https://*.oath.com https://*.ya..."".', url='https://finance.yahoo.com/sectors/energy/articles/nrc-renews-license-duke-energy-141832333.html'"
64,http://www.minnpost.com/energy/2026/04/new-thermal-battery-could-help-minnesota-campus-electrify-heat/,429.0,rate_limited,HTTP 429
89,https://www.yahoo.com/news/articles/stein-power-demand-rises-north-211542291.html,429.0,rate_limited,HTTP 429
94,https://www.wlfi.com/news/lafayette-cleanup-underway-after-storms-leave-hundreds-without-power/article_4efb86af-befd-4084-988b-cbaa6b004657.html,404.0,not_found,HTTP 404
96,https://wwmt.com/news/local/consumers-energy-deploys-500-crews-as-outages-peak-at-86000-most-back-by-tuesday-night-greg-salisbury-emmett-urbandale-pennfield-township-power-grid,404.0,not_found,HTTP 404


In [ ]:
results_df.columns

Index(['success', 'url', 'status_code', 'error_type', 'error_message',
       'fetch_method', 'extractor', 'title', 'author', 'date', 'text',
       'text_length', 'retrieved_at'],
      dtype='object')

### Test Pipeline Scraped Results

In [24]:
data_path = (ROOT 
             / 'data'
             / 'silver'
             / 'source=gdelt'
             /  'articles.parquet'
             )

In [25]:
df = pd.read_parquet(data_path)

In [26]:
valid_df = df.loc[
    df["title"].notna()
    & df["title"].str.strip().ne("")
]

duplicate_titles = valid_df["title"].value_counts()
duplicate_titles.head(10)

title
Bitter cold grips the eastern U.S. as storm deaths rise and power outages linger                     102
A North Carolina town is suing utility Duke Energy over climate change                                82
Southern US thaws from storm, but black ice risks remain as temperatures fluctuate                    59
FBI seizes documents from Bolton's office; Judge ties record | Hot off the Wire podcast               54
States across the wildfire-prone Western US are using AI for early detection                          47
Trump is forcing coal plants to stay open. It could cost customers billions                           45
Hegseth fights to keep Defense nomination. And, police search for gunman who shot CEO                 42
Energy Department offers $1.6 billion loan guarantee to upgrade transmission lines across Midwest     41
BizToc                                                                                                36
Governors' Wind Energy Coalition                 

In [27]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata', 'raw',
       'success', 'scrape_status', 'status_code', 'error_type',
       'error_message', 'fetch_method', 'fallback_used', 'extractor', 'author',
       'date', 'text_length', 'attempt_count'],
      dtype='object')

In [28]:
for i in df['text'].head(4):
    print(i)

Ranchers discuss lasting impacts from wildfires, status of Xcel claims
CANADIAN, Texas (KFDA) - Ranchers came together today for a beef conference to discuss lasting impacts from the 2024 wildfires in the Panhandle.
At today’s news conference, the founder and managing partner of Farm and Ranch Loss Group presented findings they claim Xcel has undervalued and failed to compensate ranchers after the fires.
“Xcel Energy accepted responsibility for the fire. They agreed to make the ranchers whole and they’ve done anything but. They’ve delayed the claims. They have referred all the claims negotiations to a New York litigation firm. There’s no contact with Xcel anymore. Now all the contact comes from the New York litigator,” said Bram Browder.
We reached out to Xcel Energy. The company provided a statement, saying “Our goal is to settle claims as quickly as possible, and we look forward to achieving a resolution. We will continue to work with landowners to reach settlements whenever possible

In [33]:
df['text'].str.strip().ne("").sum() / len(df)

np.float64(0.6783349467570184)

In [34]:
df.head(5)

,record_id,source,source_type,title,text,published_at,retrieved_at,url,region,categories,...,status_code,error_type,error_message,fetch_method,fallback_used,extractor,author,date,text_length,attempt_count
0,20250501011500-920,gdelt,api,Ranchers discuss lasting impacts from wildfire...,Ranchers discuss lasting impacts from wildfire...,2025-05-01T01:15:00+00:00,2026-06-19T23:25:56.037354+00:00,https://www.newschannel10.com/2025/04/30/ranch...,US-USNY,"[AGRICULTURE, NATURAL_DISASTER, NATURAL_DISAST...",...,200.0,None,None,aiohttp,False,trafilatura_after_aiohttp,Paige Stockton,2025-04-30,2155,1
1,20250501013000-742,gdelt,api,None,,2025-05-01T01:30:00+00:00,2026-06-19T23:25:56.739108+00:00,https://www.yahoo.com/news/gas-leak-closes-par...,None,[nan],...,429.0,rate_limited,HTTP 429,aiohttp,False,None,None,None,0,1
2,20250501024500-604,gdelt,api,Xcel Energy offers bill assistance as Minnesot...,"Apr 30, 2025\nXcel Energy offers bill assistan...",2025-05-01T02:45:00+00:00,2026-06-19T23:25:56.834214+00:00,https://www.kvsc.org/xcel-energy-offers-bill-a...,US,"[TAX_FNCACT, TAX_FNCACT_DIRECTOR, CRISISLEX_O0...",...,200.0,None,None,aiohttp,False,trafilatura_after_aiohttp,Shay Lelonek,2025-04-30,942,1
3,20250501050000-193,gdelt,api,"WisBusiness: the Show with Brent Ridge, Dairyl...",Contact: Tom Still or Julie Johnson at 608-442...,2025-05-01T05:00:00+00:00,2026-06-19T23:25:58.112369+00:00,https://www.wisbusiness.com/2025/wisbusiness-t...,US,"[TAX_FNCACT, TAX_FNCACT_CHIEF, TAX_FNCACT_EXEC...",...,200.0,None,None,aiohttp,False,trafilatura_after_aiohttp,Alex Moe,2025-04-30,2575,1
4,20250501050000-703,gdelt,api,Perkins Road Overpass business owners say it's...,Perkins Road Overpass business owners say it's...,2025-05-01T05:00:00+00:00,2026-06-19T23:25:58.798342+00:00,https://www.wbrz.com/news/perkins-road-overpas...,None,"[EPU_ECONOMY_HISTORIC, CRISISLEX_CRISISLEXREC,...",...,200.0,None,None,aiohttp,False,trafilatura_after_aiohttp,Brittany Weiss,2026-06-19,3433,1


In [44]:
df.loc[df['error_message'].notna(), ['error_message']].value_counts()

error_message                                                                                                                                                                                                                                                         
HTTP 404                                                                                                                                                                                                                                                                  2903
'ArticleScraperConfig' object has no attribute 'playwright_timeout_ms'                                                                                                                                                                                                    1818
HTTP 429                                                                                                                                                                                           

In [54]:
for title in np.random.choice(
    df.loc[df["title"].notna(), "title"].unique(),
    size=20,
    replace=False,
):
    print(title)

Louisiana PSC Approves 3 Controversial Gas Plants Ahead of Schedule for Meta Data Center
Letters to the Editor 3-2: Affordable healthcare and renewable energy
Consumers Energy Natural Gas Rates Increase, Request for Electric Rate Increase - WSGW 790 AM & 100.5 FM
New Alliant Energy Plant Could Interfere With Eastern Iowa Airport Flights | NewsRadio 600 WMT
CenterPoint Energy’s EPS Forecast Prompts Jefferies To Lift Price Target
Arkansas Nuclear One recognized 50 years of operation in December
SFFR responds to structure fire in Sioux Falls Friday afternoon
County to consider renaming Pirates Cove
AI technology expands in the Mountain West to reduce wildfires
CenterPoint Energy Provides Power Restoration Timelines by Region for Southern Indiana
Major highway reopened in Charlotte after crash takes down stop light
Guest column: My river is gone
Contractor pulls out of Fairmont-Cedars project - Hendersonville Lightning
New study raises heat island concerns as NC data center boom grows
Spri